# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a walkthrough for loading and exploring the FAIR\(^2\) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id`s. This helps to identify what data is present in the package for structured access. We will enumerate all record sets and print their fields and columns, referencing them solely by their `@id`.

In [ ]:
# List all record sets and their fields/columns by @id
print("Available record sets and fields in the dataset:")
record_set_ids = []
for rs in dataset.record_sets():
    print(f"\nRecordSet @id: {rs['@id']}")
    record_set_ids.append(rs['@id'])
    if 'field' in rs:
        fields = rs['field']
        if isinstance(fields, dict):
            fields = [fields]
        print("  Fields @id:")
        for field in fields:
            print(f"   - {field['@id']}")
            if 'column' in field:
                columns = field['column']
                if isinstance(columns, dict):
                    columns = [columns]
                for col in columns:
                    print(f"      Column @id: {col['@id']}")

## 3. Data Extraction
Load data from all detected record sets into DataFrames for analysis. Reference and use the `@id` values for each record set as discovered in the previous step.

In [ ]:
# Extract data from each record set by @id
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for RecordSet @id: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        if len(records) == 0:
            print(f"  No records found for RecordSet {record_set_id}.")
            continue
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  Columns: {df.columns.tolist()}")
        print(df.head(2))
    except Exception as e:
        print(f"  Could not load records for {record_set_id}: {e}")
    print('-' * 50)

if len(dataframes) == 0:
    print("No record sets with data available in this Croissant package.")
else:
    # Choose the first available record set as the main example for further steps
    main_record_set_id = next(iter(dataframes))
    print(f"Using RecordSet @id '{main_record_set_id}' for further EDA and visualization.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data. You must use column or field `@id`s (not display names) in all manipulations.

Below, we:
- Select the first numeric field available in the main RecordSet (if any),
- Filter for values greater than a threshold,
- Normalize these values,
- Group by another field (if categorical fields are available).

Please check the field/column `@id`s from the data extraction output above.

In [ ]:
# Identify numeric fields in the main record set DataFrame (if any)
if 'main_record_set_id' not in locals():
    print("No dataframes available to run EDA.")
else:
    df = dataframes[main_record_set_id]
    numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
    if len(numeric_cols) == 0:
        print("No numeric columns found in the chosen record set.")
    else:
        # Use the first numeric column for illustration
        numeric_field_id = numeric_cols[0]
        print(f"Using numeric field @id '{numeric_field_id}' for EDA.")
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].std() > 0 else 0

        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize
        filtered_df = filtered_df.copy()
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
            if filtered_df[numeric_field_id].std() > 0 else 0
        )
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping by a possible categorical field (non-numeric, if exists)
        group_fields = [col for col in df.columns if col != numeric_field_id and df[col].dtype == 'object']
        group_field = group_fields[0] if group_fields else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
            print(f"\nGrouped data by '{group_field}':")
            print(grouped_df.head())

## 5. Visualization
Visualize the distribution of the selected numeric field and, if available, its relationship with a grouped categorical field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'main_record_set_id' in locals() and len(numeric_cols) > 0:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    # If we grouped above, show bar plot for means by group
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(10,4))
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        sns.barplot(x=group_field, y=numeric_field_id, data=grouped_df)
        plt.title(f"Mean {numeric_field_id} by {group_field}")
        plt.xticks(rotation=60)
        plt.show()

## 6. Conclusion
This notebook demonstrated how to discover, extract, and visualize data solely by referencing Croissant entity `@id`s using the `mlcroissant` library.

- Always use `@id` as a stable reference for columns/fields in programmatic data analysis workflows.
- With `mlcroissant`, you can easily extend this template for new datasets described by Croissant schemas, including automated feature exploration and validation.

Explore additional documentation at https://mlcommons.github.io/croissant/ to further harness FAIR dataset schemas in your ML pipelines.